In [1]:
from sedona.spark import *
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting Spark log level to "WARN".
26/03/16 11:48:37 INFO core/src/lib.rs: Sedona native acceleration engine v0.12.0 ready
                                                                                

In [2]:
database = 'gde_silver'sedona.sql("""
SELECT 
    COUNT(*) as total,
    COUNT(geometry) as non_null_geom,
    COUNT(high_achievement_percent) as non_null_achievement
FROM org_catalog.gde_silver.school_data
""").show()

In [3]:
sedona.sql("""
UPDATE org_catalog.gde_bronze.schools_bronze
SET geometry = ST_Point(GeoCoded_X, GeoCoded_Y)
""").show()

26/03/16 11:48:45 WARN ExecutorPodsLifecycleManager: 1 new failed executors./ 1]
26/03/16 11:48:45 ERROR TaskSchedulerImpl: Lost executor 7 on 10.1.98.19: 
The executor with id 7 exited with exit code 1(generic, look at logs to clarify).



The API gave the following container statuses:


	 container name: spark-kubernetes-executor
	 container image: registry.depot.dev/8jjbrshl5f:wherobots-spark-v2.1.13-db-22841738946
	 container state: terminated
	 container started at: 2026-03-16T11:48:41Z
	 container finished at: 2026-03-16T11:48:44Z
	 exit code: 1
	 termination reason: Error
      
                                                                                

++
||
++
++



In [4]:
# Join Census Data to Polygons

sedona.sql(f'''
create or replace table org_catalog.{database}.census_data as
    select
        a.geometry,
        a.GEOID as geoid,
        b._c2 as total_pop, 
        c._c2 as median_age,
        d._c2 as median_income
    from org_catalog.gde_bronze.block_groups_bronze a
    join org_catalog.gde_bronze.total_pop_bronze b
        on right(b._c0, 12) = a.GEOID
    join org_catalog.gde_bronze.median_age_bronze c
        on right(c._c0, 12) = a.GEOID
    join org_catalog.gde_bronze.median_income_bronze d
        on right(d._c0, 12) = a.GEOID
''')

DataFrame[]

In [5]:
# Join School Data to School Points
# Null strings in _c18 and _c21, using try_cast instead
sedona.sql(f'''
create or replace table org_catalog.{database}.school_data as

with one as (
    select 
        _c3 as ESDName, 
        _c5 as DistrictCode,
        _c8 as SchoolCode,
        _c9 as SchoolName,
        sum(try_cast(_c18 as double)) as total_students,
        sum(try_cast(_c21 as double)) as high
    from org_catalog.gde_bronze.report_card_bronze
    where 
        _c13 = 'All Students' and 
        _c14 = 'All Grades'
    group by 1, 2, 3, 4
)

select
    a.geometry,
    a.ESDName,
    a.School,
    b.SchoolCode,
    b.SchoolName,
    b.high/b.total_students as high_achievement_percent
from org_catalog.gde_bronze.schools_bronze a
join one b
    on a.ESDName = b.ESDName and 
    a.School = b.SchoolName
''')

DataFrame[]

In [6]:
sedona.sql(f"SHOW TABLES IN org_catalog.{database}").show()

+----------+--------------------+-----------+
| namespace|           tableName|isTemporary|
+----------+--------------------+-----------+
|gde_silver|         census_data|      false|
|gde_silver|       homes_buffers|      false|
|gde_silver|  homes_demographics|      false|
|gde_silver|homes_distance_to...|      false|
|gde_silver|homes_distance_to...|      false|
|gde_silver| homes_flood_hazards|      false|
|gde_silver| homes_school_scores|      false|
|gde_silver|homes_seismic_haz...|      false|
|gde_silver|homes_zoning_over...|      false|
|gde_silver|house_sales_ndvi_...|      false|
|gde_silver|  house_sales_silver|      false|
|gde_silver|house_sales_tri_s...|      false|
|gde_silver|house_sales_zonal...|      false|
|gde_silver|residential_build...|      false|
|gde_silver|residential_build...|      false|
|gde_silver|     roads_proximity|      false|
|gde_silver|         school_data|      false|
|gde_silver|     zoning_overlaps|      false|
+----------+--------------------+-

In [7]:
sedona.sql("""
SELECT 
    COUNT(*) as total,
    COUNT(geometry) as non_null_geom,
    COUNT(high_achievement_percent) as non_null_achievement
FROM org_catalog.gde_silver.school_data
""").show()

+-----+-------------+--------------------+
|total|non_null_geom|non_null_achievement|
+-----+-------------+--------------------+
|  900|          900|                 842|
+-----+-------------+--------------------+

